# 회귀 스위트

Python 3.12 새 커널에서 Before: 평균만 보는 기준의 누락부터 확인합니다.

## Python 3.12 패키지 버전 고정

In [1]:
# 실습 환경에서 사용하는 보조 패키지 버전을 고정합니다.
%pip install -q python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## Before: 평균만 보는 기준의 누락

In [2]:
# 평균 기준만 사용했을 때 Q090의 핵심 실패를 놓치는지 확인합니다.
baseline = {'Q002': 1, 'Q006': 1, 'Q090': 1}
current = {'Q002': 1, 'Q006': 1, 'Q090': 0}
average_pass = sum(current.values()) / len(current) >= 0.66
print('Before 평균 기준 통과:', average_pass)
print('놓친 핵심 실패:', [qid for qid in baseline if current[qid] < baseline[qid]])

Before 평균 기준 통과: True
놓친 핵심 실패: ['Q090']


## After: 사례별 품질·지연·비용 통과 조건

In [3]:
# 사례별 품질과 지연·비용 상한을 모두 검사하는 gate 함수를 정의합니다.
limits = {'p95_total_ms': 1500, 'p99_cost_usd': 0.08}
observed = {'p95_total_ms': 980, 'p99_cost_usd': 0.06}
def gate(scores: dict[str, int]) -> bool:
    quality = all(scores[qid] >= baseline[qid] for qid in baseline)
    latency = observed['p95_total_ms'] <= limits['p95_total_ms']
    cost = observed['p99_cost_usd'] <= limits['p99_cost_usd']
    return quality and latency and cost
print('After 사례별 기준 통과:', gate(current))

After 사례별 기준 통과: False


## 수정 결과를 같은 기준으로 재검사

In [4]:
# Q090을 수정한 결과가 같은 고정 기준을 통과하는지 검사합니다.
fixed = {'Q002': 1, 'Q006': 1, 'Q090': 1}
print('수정 결과 통과:', gate(fixed))
assert not gate(current)
assert gate(fixed)
print('회귀 데이터셋 버전: rag-regression-v1.0.0')

수정 결과 통과: True
회귀 데이터셋 버전: rag-regression-v1.0.0
